In [4]:
from pyspark import SparkContext

In [5]:
from pyspark.sql import SparkSession

In [6]:
spark = SparkSession.builder.getOrCreate()

In [7]:
df = spark.read.json("transactions_10k.jsonl") 

In [13]:
#1. Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.
from pyspark.sql.functions import col, window, avg, round

gdansk_worst_hour = (
    df.filter(col("store") == "Gdańsk")            
      .groupBy(window("timestamp", "1 hour"))
      .agg(round(avg("amount"), 2).alias("srednia_PLN"))
      .orderBy("srednia_PLN")
)

gdansk_worst_hour.show(truncate=False)

#odp: 08:00 - 09:00

+------------------------------------------+-----------+
|window                                    |srednia_PLN|
+------------------------------------------+-----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.01     |
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|412.92     |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|415.91     |
+------------------------------------------+-----------+



In [11]:
#2.Policz ile transakcji per kategoria było w oknie 09:00–09:30.
from pyspark.sql.functions import sum as _sum, min as _min, max as _max, count
category_summary = (
    df.groupBy("category", window("timestamp", "30 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
    )
    .filter(col("window.start") == "2026-04-12 09:00:00")
    .orderBy("window","category")
)
category_summary.show()

#odp: elektronika - 611, książki - 622, odzież - 605, żywność - 567

+-----------+--------------------+---------+
|   category|              window|liczba_tx|
+-----------+--------------------+---------+
|elektronika|{2026-04-12 09:00...|      611|
|    książki|{2026-04-12 09:00...|      622|
|     odzież|{2026-04-12 09:00...|      605|
|    żywność|{2026-04-12 09:00...|      567|
+-----------+--------------------+---------+



In [12]:
#3.Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).
from pyspark.sql.functions import window, desc, count

cwiercgodz = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
    )
    .orderBy(desc("liczba_tx"))
)
cwiercgodz.show(truncate=False)

#odp: 09:15 - 09:30

+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |
|{2026-04-12 09:00:00, 2026-04-12 09:15:00}|1171     |
|{2026-04-12 09:30:00, 2026-04-12 09:45:00}|1156     |
|{2026-04-12 08:45:00, 2026-04-12 09:00:00}|1139     |
|{2026-04-12 09:45:00, 2026-04-12 10:00:00}|1100     |
|{2026-04-12 08:30:00, 2026-04-12 08:45:00}|899      |
|{2026-04-12 10:00:00, 2026-04-12 10:15:00}|858      |
|{2026-04-12 08:15:00, 2026-04-12 08:30:00}|644      |
|{2026-04-12 10:15:00, 2026-04-12 10:30:00}|582      |
|{2026-04-12 08:00:00, 2026-04-12 08:15:00}|468      |
|{2026-04-12 10:30:00, 2026-04-12 10:45:00}|443      |
|{2026-04-12 10:45:00, 2026-04-12 11:00:00}|306      |
+------------------------------------------+---------+

